**Imports**

In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

**Model Architecture**

In [10]:
class SmallCNN(nn.Module):
    def __init__(self,num_classes = 9):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1,out_channels=32,kernel_size=3,stride=1,padding=1)
        self.conv2 = nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,stride=1,padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2,stride=2)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(in_features=64*7*7,out_features=128)
        self.fc2 = nn.Linear(in_features=128,out_features=num_classes)
        self.dropout = nn.Dropout(0.25)
        
    def forward(self,X,return_all_feats=False):
        f1 = self.pool(self.relu(self.conv1(X)))
        f2 = self.pool(self.relu(self.conv2(f1)))
        f2_flat = f2.flatten(1)
        f3 = self.relu(self.fc1(f2_flat))
        out = self.fc2(self.dropout(f3))
        
        if return_all_feats:
            # [N,C,H,W] -> [N,C]
            f1_pooled = f1.mean(dim=[2,3])
            f2_pooled = f2.mean(dim=[2,3])
            return out,[f1_pooled,f2_pooled]
        return out

**Fit Class Gaussians on features**

In [11]:
@torch.inference_mode()
def compute_class_stats(model,train_loader,num_classes,device):
    model.eval()
    
    all_feats,all_labels = None,[]
    
    for X,y in train_loader:
        X = X.to(device)
        out,feats_list = model(X,return_all_feats=True)
        
        if all_feats is None:
            all_feats = [[] for _ in feats_list]
        for l,f in enumerate(feats_list):
            all_feats[l].append(f.cpu())
        all_labels.append(y)
    
    all_labels = torch.cat(all_labels)
    all_feats = [torch.cat(feat) for feat in all_feats]
    
    stats = [] #[(class_means),inv_cov]
    for feats in all_feats:
        feat_dim = feats.shape[1]
        
        class_means = torch.stack([feats[all_labels==c].mean(dim=0) for c in range(num_classes)])
        centered = torch.cat([feats[all_labels==c] - class_means[c] for c in range(num_classes)])
        
        cov = (centered.T @ centered) / centered.shape[0]
        cov+=1e-6*torch.eye(feat_dim)
        
        precision = torch.linalg.inv(cov)
        stats.append((class_means,precision))
    return stats

**Mahalanobis Score for batch of features**

In [12]:
def mahalanobis_score_batch(feats,class_means,precision):
    # feats of some particular layer
    
    scores_per_class = []
    for c in range(class_means.shape[0]):
        diff = feats - class_means[c]
        left = diff @ precision
        m_sq = (left*diff).sum(dim=1)
        scores_per_class.append(-m_sq)
    return torch.stack(scores_per_class,dim=1).max(dim=1).values

**Create modified input**

In [13]:
def perturbed_input(model,x,class_means,precision,layer_idx,eps,device):
    x = x.to(device).requires_grad_(True)
    
    out,feats_list = model(x,return_all_feats=True)
    feats = feats_list[layer_idx]

    score = mahalanobis_score_batch(feats,class_means.to(device),precision.to(device)).sum()
    score.backward()
    
    x_perturbed = x.detach() + eps*x.grad.detach().sign()
    return x_perturbed

In [14]:
def mahalanobis_ood_score(model,loader,stats,device,eps=0,layer_weights=None,is_labelled=True):
    model.eval()
    num_layers = len(stats)
    
    if layer_weights is None:
        layer_weights = [1/num_layers]*num_layers
    
    all_scores = []
    
    for batch in loader:
        x = batch[0] if is_labelled else batch
        x = x.to(device)
        
        layer_scores = []
        for l,(class_means,precision) in enumerate(stats):
            if eps>0:
                x_in = perturbed_input(model,x,class_means,precision,l,eps,device)
            else:
                x_in = x
        
            with torch.no_grad():
                out,feats_list = model(x_in,return_all_feats=True)
                feats = feats_list[l].cpu()
            
            score = mahalanobis_score_batch(feats,class_means,precision)
            layer_scores.append(score)
        final = sum(w*s for w,s in zip(layer_weights,layer_scores))
        all_scores.append(final)
    return torch.cat(all_scores)

**Evaluation**

In [15]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

model = SmallCNN(num_classes=9).to(device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Standard MNIST mean/std
])


class FilteredMNIST(Dataset):
    def __init__(self, base_dataset, keep_digits, remap_labels=False):
        self.dataset = base_dataset
        self.remap_labels = remap_labels
        
        mask = torch.isin(self.dataset.targets, torch.tensor(keep_digits))
        self.indices = mask.nonzero(as_tuple=True)[0]
        
    def __len__(self):
        return len(self.indices)
        
    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.remap_labels:
            label = label - 1
        return img, label

class PureNoiseDataset(Dataset):
    def __init__(self, num_samples=1000, shape=(1, 28, 28)):
        self.data = torch.randn(num_samples, *shape)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        return self.data[idx]

import os
data_dir = os.path.expanduser('~/.pytorch/data') 

mnist_train_full = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)
mnist_test_full  = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)
fashion_test     = datasets.FashionMNIST(root=data_dir, train=False, download=True, transform=transform)



id_train_ds = FilteredMNIST(mnist_train_full, keep_digits=list(range(1, 10)), remap_labels=True)
id_test_ds  = FilteredMNIST(mnist_test_full,  keep_digits=list(range(1, 10)), remap_labels=True)

zero_test_ds = FilteredMNIST(mnist_test_full, keep_digits=[0], remap_labels=False)

noise_ds = PureNoiseDataset(num_samples=10000, shape=(1, 28, 28))

batch_size = 256

train_loader   = DataLoader(id_train_ds, batch_size=batch_size, shuffle=True)
id_test_loader = DataLoader(id_test_ds,  batch_size=batch_size, shuffle=False)
zero_loader    = DataLoader(zero_test_ds, batch_size=batch_size, shuffle=False)
fashion_loader = DataLoader(fashion_test, batch_size=batch_size, shuffle=False)
noise_loader   = DataLoader(noise_ds,     batch_size=batch_size, shuffle=False)


In [16]:
def auroc(id_scores,ood_scores):
    # Mahalanobis score is higher for ID, so negated for OOD convention
    y_true  = [0] * len(id_scores) + [1] * len(ood_scores)
    y_score = torch.cat([-id_scores, -ood_scores]).numpy()
    return roc_auc_score(y_true, y_score)


# Fit stats on training data
stats = compute_class_stats(model, train_loader, num_classes=9, device=device)

# Sweep eps
print("Sweeping eps (no ensemble, last layer only):\n")
best_eps = 0
best_auroc = 0
for eps in [0.0, 0.001, 0.002, 0.005, 0.01]:
    
    # Isolate the last layer by setting earlier layer weights to 0.0
    # Use [0.0, 0.0, 1.0] if your model returns 3 features, or [0.0, 1.0] if it returns 2.
    weights = [0.0, 0.0, 1.0] if len(stats) == 3 else [0.0, 1.0]

    # Pass the FULL stats list, but apply the weights mask
    id_s   = mahalanobis_ood_score(model, id_test_loader, stats, device, eps=eps, layer_weights=weights)
    zero_s = mahalanobis_ood_score(model, zero_loader,    stats, device, eps=eps, layer_weights=weights)
    
    au_roc = auroc(id_s,zero_s)
    if au_roc>=best_auroc:
        best_auroc = au_roc
        best_eps = eps

    print(f"eps={eps:.3f}  AUROC vs digit 0: {auroc(id_s, zero_s):.4f}")


print()

# Final evaluation: ensemble all 3 layers, best eps
id_s      = mahalanobis_ood_score(model, id_test_loader, stats, device, eps=best_eps)
zero_s    = mahalanobis_ood_score(model, zero_loader,    stats, device, eps=best_eps)
fashion_s = mahalanobis_ood_score(model, fashion_loader, stats, device, eps=best_eps)
noise_s   = mahalanobis_ood_score(model, noise_loader,   stats, device, eps=best_eps, is_labelled=False)

print(f"Mahalanobis (eps={best_eps}, 3-layer ensemble)")
print(f"  AUROC vs digit 0:      {auroc(id_s, zero_s):.4f}")
print(f"  AUROC vs FashionMNIST: {auroc(id_s, fashion_s):.4f}")
print(f"  AUROC vs noise:        {auroc(id_s, noise_s):.4f}")

Sweeping eps (no ensemble, last layer only):

eps=0.000  AUROC vs digit 0: 0.8141
eps=0.001  AUROC vs digit 0: 0.8136
eps=0.002  AUROC vs digit 0: 0.8129
eps=0.005  AUROC vs digit 0: 0.8106
eps=0.010  AUROC vs digit 0: 0.8054

Mahalanobis (eps=0.0, 3-layer ensemble)
  AUROC vs digit 0:      0.7993
  AUROC vs FashionMNIST: 0.9999
  AUROC vs noise:        1.0000


**Analysis**

FashionMNIST and noise being very different from MNIST are detected to a very good accuracy
but 0 which is Near-OOD has a lower AUROC

Why non-zero eps hurts the performance :
Since 0 is a case of Near OOD,it has several features similar to other digits and the perturbation ends up adding uncertainity to the features instead of clarifying and AUROC drops

Why 3-Layer Ensemble performed worse :
Early Layers have features very similar across all digits and the penultimate layer has much better differentiable features instead - the ensembling ended up diluting their power into the final prediction with uniform weights for each layer
This shows the need for training the weights for these layers as well (as discussed in the paper)